# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
# Setup for the fixed FlyRank warehouse release.
# The token is read at runtime; never paste it into the notebook source.

import os
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add a Hugging Face READ token as a Colab Secret named HF_TOKEN."
    )

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)
APRIL = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-04/*.parquet'"
    f")"
)

print("Warehouse paths ready.")
print("Feature window: March 1-31, 2026")
print("Outcome window: April 1-30, 2026")


Warehouse paths ready.
Feature window: March 1-31, 2026
Outcome window: April 1-30, 2026


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# EXACTLY THREE formal verification queries for the Assignment 4 contract.

# Query 1 — raw grain: zero rows means no duplicate
# (report_date, client_hash_id, content_hash_id) keys.
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {MARCH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("QUERY 1 — grain check")
display(grain_check)


# Query 2 — size and date span of the March feature partition.
march_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS pages,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {MARCH}
""").df()

print("\nQUERY 2 — March size and date span")
display(march_check)


# Query 3 — availability. Use IS TRUE because availability flags are
# not safely treated as ordinary two-valued booleans.
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM {MARCH}
""").df()

print("\nQUERY 3 — March availability")
display(availability_check)


QUERY 1 — grain check


,report_date,client_hash_id,content_hash_id,row_count



QUERY 2 — March size and date span


,total_rows,clients,pages,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31



QUERY 3 — March availability


,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


### Design analysis 1 — determine the minimum usable-day requirement

This analysis is **not a fourth formal verification query**. It is a design-analysis block used to determine the coverage rule from the exact `FlyRank/internship-warehouse` March and April partitions.

A **usable GSC day** means a day for which the page has a daily record with `gsc_data_available IS TRUE`. A page may have zero impressions on a valid observed day, so usable-day coverage is **not** defined as `gsc_impressions > 0`.

No minimum-day value is assumed in advance.

The procedure is:

1. measure the March usable-day distribution at page level;
2. compare several candidate March cutoffs;
3. measure how many clients and pages each cutoff retains;
4. independently inspect April outcome-window coverage;
5. choose a practical March feature-eligibility cutoff from the observed trade-off;
6. choose an April outcome-observability requirement separately.

The March rule defines whether a page has enough **past information** to enter the proof-of-concept population. The April rule does **not** determine which pages are sampled in March. It only determines whether a selected March page has enough future observations for its April outcome to be evaluated reliably.

After the cells below are executed on the exact warehouse, the selected values and the numerical evidence supporting them will be written here explicitly.


In [4]:
# STEP 1A — exact March page-level GSC coverage.
# This scans only the March partition of FlyRank/internship-warehouse.

march_coverage = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS march_usable_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Pages with at least one usable March GSC day: {len(march_coverage):,}")
print(f"Clients represented: {march_coverage['client_hash_id'].nunique():,}")

print("\nMarch usable-day distribution summary:")
display(
    march_coverage["march_usable_days"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
    .to_frame("march_usable_days")
)

print("\nExact frequency table:")
march_day_distribution = (
    march_coverage["march_usable_days"]
    .value_counts()
    .sort_index()
    .rename_axis("usable_days")
    .reset_index(name="pages")
)
march_day_distribution["share_pct"] = (
    100.0 * march_day_distribution["pages"] / len(march_coverage)
)
display(march_day_distribution)


Pages with at least one usable March GSC day: 176,738
Clients represented: 47

March usable-day distribution summary:


,march_usable_days
count,176738.000000
mean,20.431718
std,11.480153
min,1.000000
10%,2.000000
25%,9.000000
50%,26.000000
75%,31.000000
90%,31.000000
max,31.000000



Exact frequency table:


,usable_days,pages,share_pct
0,1,13321,7.537145
1,2,7723,4.369745
2,3,5347,3.025382
3,4,4359,2.466363
4,5,3658,2.069730
5,6,3515,1.988820
6,7,2926,1.655558
7,8,2785,1.575779
8,9,2585,1.462617
9,10,2229,1.261189


In [5]:
# STEP 1B — compare candidate March cutoffs.
# No cutoff is locked before this table is inspected.

candidate_cutoffs = [7, 14, 18, 20, 21, 24, 27, 28, 30, 31]

rows = []
for cutoff in candidate_cutoffs:
    eligible = march_coverage[
        march_coverage["march_usable_days"] >= cutoff
    ].copy()

    per_client = eligible.groupby("client_hash_id").size()

    rows.append({
        "minimum_march_days": cutoff,
        "pages_retained": len(eligible),
        "clients_retained": eligible["client_hash_id"].nunique(),
        "pct_observed_pages_retained": round(
            100.0 * len(eligible) / len(march_coverage), 2
        ),
        "median_pages_per_client": (
            float(per_client.median()) if len(per_client) else 0.0
        ),
        "smallest_client_page_count": (
            int(per_client.min()) if len(per_client) else 0
        ),
        "pct_of_march_window_required": round(
            100.0 * cutoff / 31, 2
        )
    })

march_cutoff_analysis = pd.DataFrame(rows)
display(march_cutoff_analysis)


,minimum_march_days,pages_retained,clients_retained,pct_observed_pages_retained,median_pages_per_client,smallest_client_page_count,pct_of_march_window_required
0,7,138815,43,78.54,987.0,1,22.58
1,14,121844,40,68.94,738.5,1,45.16
2,18,111162,39,62.90,571.0,1,58.06
3,20,106546,37,60.28,713.0,1,64.52
4,21,103225,37,58.41,678.0,1,67.74
5,24,96013,37,54.33,589.0,1,77.42
6,27,86742,37,49.08,504.0,1,87.10
7,28,83842,37,47.44,478.0,1,90.32
8,30,67880,36,38.41,234.5,1,96.77
9,31,61796,34,34.96,295.0,1,100.00


In [6]:
# STEP 1C — inspect April outcome observability separately.
# IMPORTANT: April information is NOT used to construct the March sample.
# It is used only to determine whether a future outcome can be evaluated.

april_coverage = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS april_usable_days
    FROM {APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Pages with at least one usable April GSC day: {len(april_coverage):,}")
print(f"April clients represented: {april_coverage['client_hash_id'].nunique():,}")

print("\nApril usable-day distribution summary:")
display(
    april_coverage["april_usable_days"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
    .to_frame("april_usable_days")
)

coverage_pair = march_coverage.merge(
    april_coverage,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)
coverage_pair["april_usable_days"] = (
    coverage_pair["april_usable_days"].fillna(0).astype(int)
)

joint_rows = []
for cutoff in candidate_cutoffs:
    march_selected = coverage_pair[
        coverage_pair["march_usable_days"] >= cutoff
    ]
    outcome_observable = march_selected[
        march_selected["april_usable_days"] >= cutoff
    ]

    joint_rows.append({
        "candidate_days": cutoff,
        "march_feature_eligible_pages": len(march_selected),
        "march_feature_eligible_clients": march_selected["client_hash_id"].nunique(),
        "of_those_april_observable_pages": len(outcome_observable),
        "of_those_april_observable_clients": outcome_observable["client_hash_id"].nunique(),
        "pct_of_march_selected_with_observable_april": round(
            100.0 * len(outcome_observable) / len(march_selected), 2
        ) if len(march_selected) else 0.0
    })

coverage_tradeoff = pd.DataFrame(joint_rows)
display(coverage_tradeoff)

print(
    "\nInterpretation rule: choose the March eligibility cutoff from March coverage; "
    "treat April coverage only as label/outcome observability, not as a March sampling filter."
)


Pages with at least one usable April GSC day: 194,760
April clients represented: 51

April usable-day distribution summary:


,april_usable_days
count,194760.000000
mean,20.030088
std,10.985802
min,1.000000
10%,2.000000
25%,9.000000
50%,25.000000
75%,30.000000
90%,30.000000
max,30.000000


,candidate_days,march_feature_eligible_pages,march_feature_eligible_clients,of_those_april_observable_pages,of_those_april_observable_clients,pct_of_march_selected_with_observable_april
0,7,138815,43,128040,43,92.24
1,14,121844,40,111354,40,91.39
2,18,111162,39,100427,39,90.34
3,20,106546,37,95633,37,89.76
4,21,103225,37,92120,37,89.24
5,24,96013,37,84146,37,87.64
6,27,86742,37,73918,36,85.22
7,28,83842,37,70248,36,83.79
8,30,67880,36,52124,33,76.79
9,31,61796,34,0,0,0.00



Interpretation rule: choose the March eligibility cutoff from March coverage; treat April coverage only as label/outcome observability, not as a March sampling filter.


In [7]:
# STEP 1D — verify exact cohort identity at the chosen 20-day rule.
# This is a design check, not one of the three formal verification queries.

march20 = coverage_pair[
    coverage_pair["march_usable_days"] >= 20
].copy()

labeled20 = march20[
    march20["april_usable_days"] >= 20
].copy()

march20_clients = set(march20["client_hash_id"])
labeled20_clients = set(labeled20["client_hash_id"])

print("March >=20 clients:", len(march20_clients))
print("April-observable clients within that March cohort:", len(labeled20_clients))
print("Exact same client IDs:", march20_clients == labeled20_clients)
print("March clients missing from labeled cohort:", len(march20_clients - labeled20_clients))
print("Unexpected April-only clients:", len(labeled20_clients - march20_clients))

march20_keys = set(zip(march20["client_hash_id"], march20["content_hash_id"]))
labeled20_keys = set(zip(labeled20["client_hash_id"], labeled20["content_hash_id"]))

print("\nMarch feature-eligible page keys:", len(march20_keys))
print("Same page keys with >=20 April days:", len(labeled20_keys))
print("Every labeled page comes from the March cohort:", labeled20_keys.issubset(march20_keys))
print("March pages without sufficient April observability:", len(march20_keys - labeled20_keys))


March >=20 clients: 37
April-observable clients within that March cohort: 37
Exact same client IDs: True
March clients missing from labeled cohort: 0
Unexpected April-only clients: 0



March feature-eligible page keys: 106546
Same page keys with >=20 April days: 95633
Every labeled page comes from the March cohort: True
March pages without sufficient April observability: 10913


### Design analysis 2 — derive absolute March exposure tiers

This analysis uses only the **95,633 matched longitudinal page IDs** established in Step 1. The exposure signal is each page's **average March GSC impressions per usable GSC day**:

[
	ext{March exposure}_i =
rac{sum_d 	ext{gsc impressions}_{i,d}}
{	ext{usable March GSC days}_i}.
]

The purpose of the Low / Medium / High exposure tiers is **sampling balance**, not to label page quality or success.

The thresholds must be:

- derived from the observed matched-cohort distribution rather than assumed beforehand;
- expressed as **absolute average-impressions-per-day cut points**;
- identical for every client;
- evaluated for whether they preserve broad client representation and leave enough pages in every tier for equal-per-client sampling.

Three distribution-based candidate schemes are compared below: pooled terciles, pooled quartile outer bands, and pooled 20th/80th-percentile outer bands. These candidates are diagnostics, not final thresholds until their client-level viability is inspected.


In [ ]:
# STEP 2A — March average impressions/day for the exact matched cohort.

matched_keys = labeled20[["client_hash_id", "content_hash_id"]].drop_duplicates().copy()
con.register("matched_keys", matched_keys)

march_exposure = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        COUNT(DISTINCT f.report_date) AS march_usable_days,
        SUM(f.gsc_impressions) AS march_total_impressions,
        SUM(f.gsc_impressions)::DOUBLE
            / COUNT(DISTINCT f.report_date) AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN matched_keys AS k
        USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

print("Matched page rows:", f"{len(march_exposure):,}")
print("Matched clients:", march_exposure["client_hash_id"].nunique())
print(
    "Exact key count preserved:",
    len(march_exposure) == len(matched_keys)
)
print(
    "All pages have >=20 usable March days:",
    bool((march_exposure["march_usable_days"] >= 20).all())
)

print("\nMarch average-impressions/day distribution:")
display(
    march_exposure["march_avg_impressions_per_day"]
    .describe(
        percentiles=[
            0.10, 0.20, 0.25, 1/3, 0.50, 2/3,
            0.75, 0.80, 0.90, 0.95, 0.99
        ]
    )
    .to_frame("march_avg_impressions_per_day")
)

zero_pages = int(
    (march_exposure["march_avg_impressions_per_day"] == 0).sum()
)
print(
    f"Pages with zero average March impressions/day: "
    f"{zero_pages:,} "
    f"({100 * zero_pages / len(march_exposure):.2f}%)"
)


In [ ]:
# STEP 2B — create data-driven candidate absolute exposure thresholds.

x = march_exposure["march_avg_impressions_per_day"]

candidate_schemes = {
    "terciles_33_67": (
        float(x.quantile(1/3)),
        float(x.quantile(2/3))
    ),
    "quartile_25_75": (
        float(x.quantile(0.25)),
        float(x.quantile(0.75))
    ),
    "outer_20_80": (
        float(x.quantile(0.20)),
        float(x.quantile(0.80))
    )
}

threshold_rows = []
for scheme, (low_thr, high_thr) in candidate_schemes.items():
    tier = pd.cut(
        x,
        bins=[-float("inf"), low_thr, high_thr, float("inf")],
        labels=["Low", "Medium", "High"],
        include_lowest=True
    )

    counts = tier.value_counts().reindex(["Low", "Medium", "High"], fill_value=0)

    threshold_rows.append({
        "scheme": scheme,
        "low_max_avg_impressions_per_day": low_thr,
        "high_min_boundary_avg_impressions_per_day": high_thr,
        "low_pages": int(counts["Low"]),
        "medium_pages": int(counts["Medium"]),
        "high_pages": int(counts["High"]),
        "low_pct": round(100 * counts["Low"] / len(x), 2),
        "medium_pct": round(100 * counts["Medium"] / len(x), 2),
        "high_pct": round(100 * counts["High"] / len(x), 2)
    })

threshold_comparison = pd.DataFrame(threshold_rows)
display(threshold_comparison)


In [ ]:
# STEP 2C — client-level viability of each candidate threshold scheme.
# For equal-per-client sampling, a client must contribute pages to all three tiers.

viability_rows = []

for scheme, (low_thr, high_thr) in candidate_schemes.items():
    temp = march_exposure[
        ["client_hash_id", "content_hash_id", "march_avg_impressions_per_day"]
    ].copy()

    temp["exposure_tier"] = pd.cut(
        temp["march_avg_impressions_per_day"],
        bins=[-float("inf"), low_thr, high_thr, float("inf")],
        labels=["Low", "Medium", "High"],
        include_lowest=True
    )

    client_tiers = (
        temp.groupby(["client_hash_id", "exposure_tier"], observed=False)
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["Low", "Medium", "High"], fill_value=0)
    )

    client_tiers["min_tier_pages"] = client_tiers[
        ["Low", "Medium", "High"]
    ].min(axis=1)

    viability_rows.append({
        "scheme": scheme,
        "clients_total": len(client_tiers),
        "clients_with_all_3_tiers": int((client_tiers["min_tier_pages"] >= 1).sum()),
        "clients_with_at_least_10_each": int((client_tiers["min_tier_pages"] >= 10).sum()),
        "clients_with_at_least_25_each": int((client_tiers["min_tier_pages"] >= 25).sum()),
        "clients_with_at_least_50_each": int((client_tiers["min_tier_pages"] >= 50).sum()),
        "clients_with_at_least_100_each": int((client_tiers["min_tier_pages"] >= 100).sum()),
        "median_of_client_min_tier_pages": float(client_tiers["min_tier_pages"].median()),
        "p25_of_client_min_tier_pages": float(client_tiers["min_tier_pages"].quantile(0.25)),
        "smallest_client_min_tier_pages": int(client_tiers["min_tier_pages"].min())
    })

viability_comparison = pd.DataFrame(viability_rows)
display(viability_comparison)

# Keep the per-client table for the pooled-tercile candidate visible as the
# strongest default balancing option unless the diagnostics contradict it.
tercile_low, tercile_high = candidate_schemes["terciles_33_67"]
tercile_page_tiers = march_exposure.copy()
tercile_page_tiers["exposure_tier"] = pd.cut(
    tercile_page_tiers["march_avg_impressions_per_day"],
    bins=[-float("inf"), tercile_low, tercile_high, float("inf")],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

tercile_client_counts = (
    tercile_page_tiers
    .groupby(["client_hash_id", "exposure_tier"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Low", "Medium", "High"], fill_value=0)
)
tercile_client_counts["min_tier_pages"] = tercile_client_counts.min(axis=1)

print("\nPooled-tercile per-client tier counts:")
display(
    tercile_client_counts
    .sort_values("min_tier_pages")
)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.